In [0]:
%sql
USE CATALOG ecommerce;
USE SCHEMA silver;

Load the data

In [0]:
df_orders = spark.read.table("ecommerce.bronze.orders")
df_orders.display()

In [0]:
df_customers = spark.read.table("ecommerce.bronze.customers")
df_customers.display()

In [0]:
df_customers.printSchema()
df_orders.printSchema()

In [0]:
from pyspark.sql.functions import *

Checking Null Values

In [0]:
df_orders.select([
    sum(col(c).isNull().cast('int')).alias(c)
    for c in df_orders.columns
]).display()

In [0]:
df_orders = df_orders.dropna(subset=["OrderID", "CustomerID"], how='any')
df_orders.display()

In [0]:
df_customers.select([
    sum(col(c).isNull().cast('int')).alias(c)
    for c in df_customers.columns
]).display()

In [0]:
df_customers = df_customers.dropna(subset=["customer_id"], how='any')
df_customers.display()

DUPLICATE CHECK

In [0]:
print("Orders Total:", df_orders.count())
print("Orders Distinct:", df_orders.dropDuplicates(["OrderID"]).count())
df_orders = df_orders.dropDuplicates(["OrderID"])

In [0]:
df_orders.display()

In [0]:
df_orders = df_orders.filter(~col("OrderID").endswith("-X"))
df_orders.display()

In [0]:
#Analyze duplicates
from pyspark.sql.functions import countDistinct

# Find customer IDs with multiple names
problem_ids = df_customers.groupBy("customer_id") \
    .agg(countDistinct("customer_name").alias("name_count")) \
    .filter("name_count > 1") \
    .select("customer_id")

# Show full records for analysis
df_customers.select("customer_id", "customer_name") \
    .join(problem_ids, "customer_id") \
    .orderBy("customer_id") \
    .display()

In [0]:
#Identify duplicates
df_customers.groupBy("customer_id").count().filter("count > 1").display()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

window_spec = Window.partitionBy("customer_id").orderBy(desc("customer_name"))

df_customers = df_customers.withColumn(
    "rn",
    row_number().over(window_spec)
).filter("rn = 1").drop("rn")

In [0]:
df_customers.display()

TRIMMING

In [0]:
from pyspark.sql.functions import trim, col
from pyspark.sql.types import StringType

In [0]:
df_orders = df_orders.select([
    trim(col(c)).alias(c) if dict(df_orders.dtypes)[c] == 'string' else col(c)
    for c in df_orders.columns
])

In [0]:
df_customers = df_customers.select([
    trim(col(c)).alias(c) if dict(df_customers.dtypes)[c] == 'string' else col(c)
    for c in df_customers.columns
])

CASE STANDARDIZATION

In [0]:
columns_to_format = ["FirstName","LastName","City","OrderSeason","OrderDayOfWeek","ProductCategory","ProductSubCategory","PaymentMethod","DeviceType","MembershipTier","Returned"]

for c in columns_to_format:
    df_orders = df_orders.withColumn(c, initcap(col(c)))

In [0]:
customers_format_cols = ["customer_name", "city"]

for c in customers_format_cols:
    df_customers = df_customers.withColumn(c, initcap(col(c)))

DATE FORMATTING

In [0]:
from pyspark.sql.functions import to_date, col

df_orders =df_orders.withColumn("OrderDate",to_date(col("OrderDate"), "dd-MM-yyyy"))

In [0]:
df_orders.display()
df_customers.display()

Numeric validation

In [0]:
from pyspark.sql.functions import col

df_orders = df_orders.filter(
    (col("Quantity") > 0) &
    (col("UnitPrice") > 0) &
    (col("ShippingDays") >= 0) &
    (col("ReviewScore").between(1, 5))
)

In [0]:
df_orders.display()

In [0]:
df_orders.display()

In [0]:
df_customers.display()

Save a table in silver schema

In [0]:
df_orders.write.mode("overwrite").saveAsTable("ecommerce.silver.orders")

In [0]:
df_customers.write.mode("overwrite").saveAsTable("ecommerce.silver.customers")